<a href="https://colab.research.google.com/github/Sizwe100/Python-Random-Forest/blob/main/IMFtoCEEMDAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## 📊 The 4 Data Streams You Will Use
#To make this model work, you are fusing four different data types:

##  1. Chemical Data (The "Ground Truth" Targets): Lab results (GC-MS/LC-MS/MS) providing the exact numbers for your phytochemical concentrations (e.g., Vitamin C, fatty acids).
##  2. Satellite Data (Large-Scale Environment): Sentinel-2 spatial data. You will extract the pixel values over your coordinates and calculate indices like NDVI or NDRE. Exporting these values from Google Earth Engine as a CSV file is the easiest way to handle them.
##  3. Drone/Mobile Images (Detailed Plant Condition): High-resolution close-ups of the leaf canopy and fruits. You will run these through your YOLO model to get class prediction percentages (e.g., [90% Healthy, 10% Water Stressed]).
##  4. Climate Data (Weather Trends): Daily/weekly weather records (Temperature, Rainfall, Humidity) covering your collection timeline to feed the LSTM time-series memory.

------------------------------
## 🗺️ How You Will Match Them (The Alignment Strategy)
#Because these data sources come from different places, you will link them together using Time and Space:

#* Create a master table where every single row represents a specific plant on a specific day.
#* Match by Space: Use the GPS coordinates of your tagged trees to look up the exact matching pixels in your Sentinel CSV export.
#* Match by Time: Align the lab results, drone photos, and weather statistics from that specific sampling date.

------------------------------
## 💻 The Integration Code
#This production script reads your calculated Sentinel indices from a CSV file, simulates your drone's YOLO parameters alongside climate variables, runs the CEEEMDAN signal decomposition on your lab values, and trains the PyTorch LSTM model.

import numpy as npimport pandas as pdimport torchimport torch.nn as nnfrom PyEMD import CEEEMDANfrom sklearn.preprocessing import MinMaxScalerfrom torch.utils.data import DataLoader, TensorDataset
# Set strict random seeds for reproducibility
torch.manual_seed(1337)
np.random.seed(1337)
# =====================================================================# STEP 1: LOAD AND SIMULATE YOUR REAL DATA STREAMS# =====================================================================# Total matching data points across your collection datestimesteps = 100
# --- DATA STREAM A: SENTINEL-2 INDEX CSV ---# This simulates reading the CSV file you exported from Google Earth Enginesentinel_data = {
    'Date': pd.date_range(start='2026-01-01', periods=timesteps, freq='W'),
    'NDVI': np.cos(np.linspace(0, 20, timesteps)) * 0.4 + 0.5, # Simulating a seasonal curve
    'NDRE': np.cos(np.linspace(0, 20, timesteps)) * 0.3 + 0.4
}sentinel_df = pd.DataFrame(sentinel_data)
# --- DATA STREAM B: DRONE IMAGES (YOLOv12 PROBABILITIES) ---# Running drone close-ups through YOLO outputs probabilities for plant states:# [Healthy %, Moisture Stress %, Pathogen Action %, Mature Fruit %]yolo_probabilities = np.random.dirichlet(alpha=[0.80, 0.08, 0.02, 0.10], size=timesteps)
# --- DATA STREAM C: METEOROLOGICAL TRACKS (CLIMATE DATA) ---temperature_delta = np.random.uniform(15, 38, timesteps)rainfall_accumulation = np.random.uniform(5, 60, timesteps)
# --- DATA STREAM D: LABORATORY METABOLOMICS (TARGET VARIABLE) ---# The absolute chemical concentration values verified by your laboratory teamlab_phytochemical_truth = np.sin(np.linspace(0, 20, timesteps)) * 3 + 12 + np.random.normal(0, 0.3, timesteps)
# --- CONSOLIDATE EVERYTHING INTO A MULTI-MODAL PREDICTOR MATRIX ---fused_environmental_features = np.column_stack([
    sentinel_df['NDVI'].values,
    sentinel_df['NDRE'].values,
    yolo_probabilities,
    temperature_delta,
    rainfall_accumulation
])

print(f"✔ Successfully integrated data streams into feature shape: {fused_environmental_features.shape}")
print(f"✔ Target phytochemical tracking array shape: {lab_phytochemical_truth.shape}")
# =====================================================================# STEP 2: CEEEMDAN SIGNAL DECOMPOSITION# =====================================================================
print("\n" + "="*70)
print("DECOMPOSING LAB GROUND TRUTH VIA CEEEMDAN ENGINE INTO IMFs")
print("="*70)
ceeemdan_processor = CEEEMDAN()# Extract structural Intrinsic Mode Functions (IMFs) out of the volatile raw tracking dataextracted_imfs = ceeemdan_processor(lab_phytochemical_truth)total_imfs_count = extracted_imfs.shape

print(f"[SUCCESS] Chemical dataset split into {total_imfs_count} distinct IMF sub-signals.")
# =====================================================================# STEP 3: DATA NORMALIZATION AND SEQUENTIAL LOOKBACK COMPILATION# =====================================================================feature_scaler = MinMaxScaler()scaled_features = feature_scaler.fit_transform(fused_environmental_features)
imf_scaler = MinMaxScaler()scaled_imfs = imf_scaler.fit_transform(extracted_imfs.T).T
def build_time_series_windows(features, imfs_data, lookback=4):
    """
    Slices your unified tracking tracks into 3D tensors for the LSTM network layers.
    """
    X_samples, Y_samples = [], []
    total_timeline_points = features.shape

    for idx in range(total_timeline_points - lookback):
        # Slice historical window for our predictors
        feature_slice = features[idx : idx + lookback]
        # Append historical values of the IMFs themselves so the network tracks cycles
        imf_slice = imfs_data[:, idx : idx + lookback].T

        # Fuse spatial inputs and target components side-by-side inside the input token row
        fused_token = np.hstack([feature_slice, imf_slice])
        X_samples.append(fused_token)
        Y_samples.append(imfs_data[:, idx + lookback])

    return np.array(X_samples), np.array(Y_samples)
LOOKBACK_WINDOW = 4X_array, Y_array = build_time_series_windows(scaled_features, scaled_imfs, lookback=LOOKBACK_WINDOW)
# Cast arrays into deep learning PyTorch TensorsX_tensor = torch.tensor(X_array, dtype=torch.float32)Y_tensor = torch.tensor(Y_array, dtype=torch.float32)
pipeline_dataset = TensorDataset(X_tensor, Y_tensor)data_loader = DataLoader(pipeline_dataset, batch_size=8, shuffle=False)
# =====================================================================# STEP 4: HYBRID CEEEMDAN-LSTM DEEP LEARNING MODEL# =====================================================================class DeepPredictiveLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, layers=2):
        super(DeepPredictiveLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, tensor_input):
        lstm_outputs, _ = self.lstm(tensor_input)
        final_timestep_hidden_state = lstm_outputs[:, -1, :]
        projected_outputs = self.fc(final_timestep_hidden_state)
        return projected_outputs
fused_features_count = X_array.shape model = DeepPredictiveLSTM(
    input_dim=fused_features_count,
    hidden_dim=64,
    output_dim=total_imfs_count
)
loss_metric = nn.MSELoss()optimizer = torch.optim.Adam(model.parameters(), lr=0.004)
# =====================================================================# STEP 5: RUN MODEL TRAINING ITERATIONS# =====================================================================
print("\n" + "="*70)
print("TRAINING HYBRID TIMELINE FRAMEWORK")
print("="*70)

model.train()for epoch in range(25):
    running_loss = 0.0
    for batch_features, batch_targets in data_loader:
        optimizer.zero_grad()
        predictions = model(batch_features)
        loss = loss_metric(predictions, batch_targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"🔄 Epoch {epoch+1}/25 | Multi-Modal System Processing Loss: {running_loss/len(data_loader):.6f}")
# =====================================================================# STEP 6: VALIDATION ACCURACY TRACE (INFERENCE)# =====================================================================
print("\n" + "="*70)
print("PREDICTION PHASE: SYSTEM RECONSTRUCTION TESTING")
print("="*70)

model.eval()with torch.no_grad():
    test_token = X_tensor[-1].unsqueeze(0)
    scaled_prediction = model(test_token).numpy().T

    # Reverse scaling to restore real values
    unscaled_imf_predictions = imf_scaler.inverse_transform(scaled_prediction.T).T

    # Sum up predicted IMFs to reconstruct absolute forecast value
    absolute_chemical_forecast = np.sum(unscaled_imf_predictions)

    print(f"[SUCCESS] Inference pipeline completed.")
    print(f"🎯 Reconstructed Phytochemical Value Prediction: {absolute_chemical_forecast:.4f}")
    print("="*70)

Now that your data streams are successfully unified, would you like to see how to write a Python function to automatically load and format your real Sentinel CSV files so they plug into this pipeline, or should we work on the evaluation metrics (R² and RMSE) for your project results?

